# Phase 1 — single-scene NDVI check

Loads `data/blocks.geojson`, picks one Oliver block, pulls the clearest July 2024
scene from `COPERNICUS/S2_SR_HARMONIZED` with the Cloud Score+ mask applied
(pixels with `cs_cdf < 0.60` masked), computes NDVI, and displays it on a geemap
map clipped to the buffered block with the outline overlaid.

**Acceptance criteria (visual check):**
- Midsummer canopy NDVI sits in **0.5–0.75** (10 m pixels blend canopy and
  inter-row cover; values near 0.9 indicate a bug).
- The spatial pattern matches ground knowledge.

In [14]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import geemap

from vigor import extract

In [15]:
extract.init_ee()

## Blocks

Polygons load in WGS84, get projected to EPSG:32611, and receive the −1 m edge
buffer. Every reduction uses the buffered geometry; blocks under 10 pixels after
buffering are dropped with a warning.

In [16]:
blocks = extract.load_blocks(PROJECT_ROOT / "data" / "blocks.geojson")
blocks.drop(columns=["geometry", "geometry_buffered"])

,block_id,site,variety,planting_year,area_ha,n_pixels
0,penticton-ub-b1,penticton,chardonnay,2024,3.306338,249
1,oliver-ub-b1,oliver,Merlot,2023,0.882325,42


In [17]:
is_oliver = (
    blocks["site"].astype(str).str.contains("oliver", case=False)
    | blocks["block_id"].astype(str).str.contains("oliver", case=False)
)
candidates = blocks[is_oliver]
if candidates.empty:
    print("WARNING: no Oliver block in blocks.geojson - falling back to the first block.\n")
    candidates = blocks

block = candidates.iloc[0]
print(
    f"Selected block: {block['block_id']} (site={block['site']}, "
    f"{block['area_ha']:.2f} ha, {block['n_pixels']} px after -1 m buffer)"
)

Selected block: oliver-ub-b1 (site=oliver, 0.88 ha, 83 px after -1 m buffer)


## Clearest July 2024 scene

"Clearest" = the scene with the most unmasked NDVI pixels over the buffered
block after the Cloud Score+ mask — scored server-side over the whole month,
no Python loop over scenes.

In [18]:
region = extract.to_ee_geometry(block["geometry_buffered"])

july = extract.masked_ndvi_collection(region, "2024-07-01", "2024-08-01")
print(f"July 2024 scenes over the block: {extract.count_scenes(july)}")

scene = extract.clearest_scene(july, region)
summary = extract.scene_summary(scene, region)

print(f"\nScene:      {summary['scene_id']}")
print(f"Date:       {summary['date']}")
print(f"valid_frac: {summary['valid_frac']:.3f}")
print(f"NDVI median (block): {summary['ndvi_median']:.3f}")
print(f"NDVI p10-p90:        {summary['ndvi_p10']:.3f} - {summary['ndvi_p90']:.3f}")

med = summary["ndvi_median"]
if 0.5 <= med <= 0.75:
    print("\nNDVI median is inside the expected 0.5-0.75 midsummer range.")
elif med > 0.85:
    print("\nWARNING: NDVI median near 0.9 - usually a masking/scaling bug, inspect the map.")
else:
    print("\nWARNING: NDVI median outside the expected 0.5-0.75 midsummer range - inspect the map.")

July 2024 scenes over the block: 24

Scene:      20240705T185921_20240705T190418_T10UGV
Date:       2024-07-05
valid_frac: 1.000
NDVI median (block): 0.486
NDVI p10-p90:        0.466 - 0.509



## Map

NDVI clipped to the buffered block, planted extent outlined in white, buffered
extent in yellow. Check that vigorous canopy reads 0.5–0.75 (green, not
saturated dark green) and that weak/strong areas line up with ground knowledge.

In [19]:
NDVI_VIS = {
    "min": 0.0,
    "max": 0.9,
    "palette": [
        "#a50026", "#d73027", "#f46d43", "#fdae61", "#fee08b",
        "#d9ef8b", "#a6d96a", "#66bd63", "#1a9850", "#006837",
    ],
}

m = geemap.Map()
m.add_basemap("Esri.WorldImagery")
m.addLayer(extract.ndvi_layer(scene, region), NDVI_VIS, f"NDVI {summary['date']}")
m.addLayer(extract.outline_layer(block["geometry"]), {}, "planted extent")
m.addLayer(
    extract.outline_layer(block["geometry_buffered"], color="yellow", width=1),
    {},
    "buffered block (-1 m)",
)
m.add_colorbar(NDVI_VIS, label="NDVI")
m.centerObject(region, 17)
m

Map(center=[49.20477113340669, -119.53791849439057], controls=(WidgetControl(options=['position', 'transparent…